# EXTRACTION OF TREE/BUSH MASK FROM DSM, 0.5 m
(c) Olha Kachalova, Version 11.11.2025

**The main purpose of this code** is to create high-resolution vegetation masks for trees and bushes, which are then used to label training data for deep learning–based classification of RGB orthophotos.
The required classes are as follows: 
 - trees (>3 m of height)
 - bushes (1 - 3 m of height)
 - other (objects <1 m of height, buildings)

The workflow was developed in **Bash** and **AWK**, employing **GDAL (v3.8.4)** and **pktools (v2.6.7)** commands for geospatial data processing:  

| gdalinfo | gdalwarp | gdalbuildvrt | gdal_translate | gdal_rasterize | gdal_edit.py | gdal_calc.py | gdaltindex | 
| ogrinfo | ogr2ogr (buffer, dissolve) |   
| pkinfo | pkgetmask | pksetmask | pkreclass | pkstat -hist | pkfillnodata | pkcomposite |  

**Input data description:**
1. (raster) **Digital Surface Model** of the Czech Republic from image correlation of aerial survey (stereo orthophoto), created 11.10.2025 from data acquired in 2024 in raster interpolated from cloud points. Resolution: 0.5 m. Open data under CC BY 4.0. https://geoportal.cuzk.gov.cz/(S(vcjhqr3ef3xeptki2uzzpfmr))/Default.aspx?lng=EN&mode=TextMeta&side=vyskopis&metadataID=CZ-CUZK-DMPOK&mapid=8&menu=3032 Data is provided in tiles according to Czech orthophoto tiling system (2500 x 2000 m in EPSG: 5514 (S-JTSK / Krovak East North)).
2. (raster) **Digital Elevation Model** of the Czech Republic 1.0 m, raster created from open LIDAR derived LAZ dataset DMR 5G (CC BY 4.0) https://geoportal.cuzk.gov.cz/(S(vcjhqr3ef3xeptki2uzzpfmr))/Default.aspx?lng=EN&mode=TextMeta&side=vyskopis&metadataID=CZ-CUZK-DMR5G-V&mapid=8&menu=302 
3. (vector) **Buildings layer** from ZABAGED® - Planimetric Components, open data under CC BY 4.0 https://geoportal.cuzk.gov.cz/(S(vcjhqr3ef3xeptki2uzzpfmr))/Default.aspx?mode=TextMeta&side=zabaged&metadataID=CZ-CUZK-ZABAGED-VP&mapid=8&head_tab=sekce-02-gp&menu=241

All input data is provided in EPSG: 5514 (S-JTSK / Krovak East North) projected coordinate system for Czechia, Slovakia.

An example of DSM tile versus orthophoto:
![dsm](pics/DSM_examp1.png)
![ortho](pics/Ortho_examp1.png)
DSM tiles extent:
![extent](pics/extent.png)

### Data preparation: 
Unzip the DSM tiles, write the CRS into the DSM metadata (it was not displayed in gdalinfo at first), and set NoData = -9999 for both the DSM and DEM.

In [9]:
%%bash 
mkdir /media/sf_SCENARE/DMP_temp
for f in /media/sf_SCENARE/DMP_OK/*.zip; do unzip -o "$f" -d DMP_temp; done
rm -r /media/sf_SCENARE/DMP_OK
ls /media/sf_SCENARE/DMP_temp/*.tif | wc -l # print the total number of DSM tiles
for f in /media/sf_SCENARE/DMP_temp/*.tif; do gdal_edit.py -a_nodata -9999 -a_srs EPSG:5514 "$f"; done #projection has not been seen in metadata, but it is 5514 (from the source)
GTIFF_SRS_SOURCE=EPSG
gdal_edit.py -a_nodata -9999 -a_srs EPSG:5514 /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif

314


Let's check DEM: min/max, CRS, Pixel size, Data type, NoData value:

In [38]:
! gdalinfo /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif
# OR (but goes very slowly!)
#! pkinfo -i /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif -mm -a_srs -dx -dy -ot -nodata #goes very long!

Driver: GTiff/GeoTIFF
Files: /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif
       /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif.aux.xml
Size is 213854, 115228
Coordinate System is:
PROJCRS["S-JTSK / Krovak East North",
    BASEGEOGCRS["S-JTSK",
        DATUM["System of the Unified Trigonometrical Cadastral Network",
            ELLIPSOID["Bessel 1841",6377397.155,299.1528128,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4156]],
    CONVERSION["Krovak East North (Greenwich)",
        METHOD["Krovak (North Orientated)",
            ID["EPSG",1041]],
        PARAMETER["Latitude of projection centre",49.5,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8811]],
        PARAMETER["Longitude of origin",24.8333333333333,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8833]],
        PARAMETER["Co-latitude of cone axis",30.2881397527778,
            ANGLEUNIT[

In [24]:
# The size of DEM file is 37 GIGA, so that wouldn't be a good idea to process it with python :)
! ls -l -h /media/sf_SCENARE/DEM*.tif

-rwxrwx--- 1 root vboxsf 37G Oct 29 16:24 /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif


DEM and DSM has different resolution (1.0 and 0.5 m respectively), and their pixel grids aren't alighed. Let's fix this:  

![pix](pics/pix_align.png)

In [14]:
%%bash
# Build a VRT of all DSM tifs snapped to its own native grid 
gdalbuildvrt -overwrite -vrtnodata -9999 -a_srs EPSG:5514 \
  /media/sf_SCENARE/DMP_temp/A.vrt /media/sf_SCENARE/DMP_temp/*.tif
pkinfo -te -i /media/sf_SCENARE/DMP_temp/A.vrt # grab vrt corners

0...10...20...30...40...50...60...70...80...90...100 - done.
-te -560000 -1210000 -487500 -1166000


In [13]:
%%bash
# Warp DEM to DSM’s CRS, projection and extent (output is vrt!)
gdalwarp -of VRT --config GTIFF_SRS_SOURCE EPSG \
  -s_srs EPSG:5514 -t_srs EPSG:5514 \
  -tr 0.5 0.5 -tap \
  -te -560000 -1210000 -487500 -1166000 \
  -r bilinear -dstnodata -9999 -overwrite \
  /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif /media/sf_SCENARE/tmp/DEM_aligned.vrt
echo "DEM_aligned.vrt is done"

Creating output file that is 145000P x 88000L.
Processing /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif [1/1] : 0Using internal nodata values (e.g. -9999) for image /media/sf_SCENARE/DEM_JMK_Zlin_mos.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
DEM_aligned.vrt is done


### Next step: creating the object height layer (DSM - DEM)
I want to keep my height layer in tiles. For this, we need to tile our DEM into same tiles as DSM. Let's create a vector file (gpkg) with tiles' outlines

In [48]:
%%bash
gdaltindex -f GPKG /media/sf_SCENARE/footprints.gpkg \
  /media/sf_SCENARE/DMP_temp/*.tif
echo "Done."

Creating new index file...
Done.


Now let's create tiled aligned DEM (filenames = tile names)

In [59]:
%%bash

VEC="/media/sf_SCENARE/footprints.gpkg"         
DEM_SRC="/media/sf_SCENARE/DEM_JMK_Zlin_mos.tif"  
SRC="/media/sf_SCENARE/tmp/DEM_aligned.vrt"       
OUTDIR="/media/sf_SCENARE/DEM_tiles"
NODATA=-9999

mkdir -p /media/sf_SCENARE/tmp "$OUTDIR"

# Collect all unique 'location' values from the shapefile (case-sensitive field name!)
mapfile -t LOCS < <(
  ogrinfo -ro -al -geom=no -q "$VEC" \
  | awk -F'= ' '/^[[:space:]]*location[[:space:]]*\(String\)[[:space:]]*=/{print $2}' \
  | sed 's/\r$//' | sort -u
)

# Crop per polygon to tiles, naming outputs from the 'location' field
for loc in "${LOCS[@]}"; do
  name="$(basename "$loc" .tif)"
  out="$OUTDIR/${name}.tif"
  sql_val=${loc//\'/\'\'}  # escape quote if any

  gdalwarp -overwrite -of GTiff \
    -cutline "$VEC" -cwhere "location = '$sql_val'" \
    -crop_to_cutline \
    -r bilinear \
    -dstnodata $NODATA \
    -co COMPRESS=LZW -co TILED=YES -co BIGTIFF=IF_SAFER \
    "$SRC" "$out"

  echo "✓ $out"
done
echo "DEM tiles done."

Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/tmp/DEM_aligned.vrt [1/1] : 0Using internal nodata values (e.g. -9999) for image /media/sf_SCENARE/tmp/DEM_aligned.vrt.
...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/DEM_tiles/BRUM50.tif
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/tmp/DEM_aligned.vrt [1/1] : 0Using internal nodata values (e.g. -9999) for image /media/sf_SCENARE/tmp/DEM_aligned.vrt.
...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/DEM_tiles/BRUM51.tif
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/tmp/DEM_aligned.vrt [1/1] : 0Using internal nodata values (e.g. -9999) for image /media/sf_SCENARE/tmp/DEM_aligned.vrt.
...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/DEM_tiles/BRUM60.tif
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/tmp/DEM_aligned.vrt [1/1] : 0Using internal n

### Computing the object height
Object height = DSM - DEM

%%bash
#Version without xargs: (change from Markdown to Code to use)  

DSM_DIR="/media/sf_SCENARE/DMP_temp"      # DTM tiles
DEM_DIR="/media/sf_SCENARE/DEM_tiles"      # DEM tiles
OUT_DIR="/media/sf_SCENARE/height"
NODATA=-9999

mkdir -p "$OUT_DIR"

shopt -s nullglob                   # extract consistent filenames
for DSM in "$DSM_DIR"/*.tif; do
  name=$(basename "$DSM" .tif)
  DEM="$DEM_DIR/$name.tif"

  if [[ ! -f "$DEM" ]]; then             # just in case
    echo "Skip $name: no matching DEM tile found"
    continue
  fi

  OUT="$OUT_DIR/${name}_h.tif"

  gdal_calc.py -A "$DSM" -B "$DEM" \
    --calc="A-B" \
    --type=Float32 \
    --NoDataValue="$NODATA" \
    --outfile="$OUT" --overwrite \
    --co COMPRESS=LZW --co TILED=YES --co BIGTIFF=IF_SAFER

  echo "✓ $OUT"
done
echo "All done"

In [1]:
%%bash
# The previous cell, but with xargs (4 processors):
DSM_DIR="/media/sf_SCENARE/DMP_temp"      # DTM tiles
DEM_DIR="/media/sf_SCENARE/DEM_tiles"     # DEM tiles
OUT_DIR="/media/sf_SCENARE/height"
NODATA=-9999

mkdir -p "$OUT_DIR"

# make variables visible inside the xargs
export DSM_DIR DEM_DIR OUT_DIR NODATA

# find all DSM tiles, strip directory and .tif → get just "name"
find "$DSM_DIR" -maxdepth 1 -type f -name '*.tif' -printf '%f\n' | \
sed 's/\.tif$//' | \
xargs -P 4 -I{} bash -c '
  name="$1"
  DSM="$DSM_DIR/$name.tif"
  DEM="$DEM_DIR/$name.tif"

  if [[ ! -f "$DEM" ]]; then
    echo "Skip $name: no matching DEM tile found"
    exit 0
  fi

  OUT="$OUT_DIR/${name}_h.tif"

  gdal_calc.py -A "$DSM" -B "$DEM" \
    --calc="A-B" \
    --type=Float32 \
    --NoDataValue="$NODATA" \
    --outfile="$OUT" --overwrite \
    --co COMPRESS=LZW --co TILED=YES --co BIGTIFF=IF_SAFER

  echo "✓ $OUT"
' _ {}

echo "All done"


ERROR! Session/line number was not unique in database. History logging moved to new session 94
0000............10101010............20202020............30303030............404040.40.........5050..5050............6060.60.60......70...70.7070.........80...8080...80.........90.90.9090..........100 - done.
✓ /media/sf_SCENARE/height/ZLIN47_h.tif
0.100 - done.
✓ /media/sf_SCENARE/height/ZLIN49_h.tif
.0100 - done.
✓ /media/sf_SCENARE/height/ZLIN48_h.tif
100 - done.
✓ /media/sf_SCENARE/height/ZLIN46_h.tif
00..10........10....1010...20.......20.20.20...30...30....3030.......40....404040.....50.......505050........60....606060.....70.......707070.......80.....808080........90.....909090..........100 - done.
✓ /media/sf_SCENARE/height/ZLIN54_h.tif
0.100 - done.
✓ /media/sf_SCENARE/height/ZLIN56_h.tif
100 - done.
100 - done.
.✓ /media/sf_SCENARE/height/ZLIN57_h.tif
✓ /media/sf_SCENARE/height/ZLIN55_h.tif
000.10............101010...20.........20203020............30303040............5040.4040.......

Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/osgeo_utils/auxiliary/gdal_argparse.py", line 226, in main
    self.doit(**kwargs)
  File "/usr/lib/python3/dist-packages/osgeo_utils/gdal_calc.py", line 887, in doit
    return Calc(**kwargs)
           ^^^^^^^^^^^^^^
  File "/usr/lib/python3/dist-packages/osgeo_utils/gdal_calc.py", line 400, in Calc
    os.remove(outfile)
OSError: [Errno 26] Text file busy: '/media/sf_SCENARE/height/BRUM91_h.tif'


✓ /media/sf_SCENARE/height/BRUM91_h.tif
300.10........10..40..20....20..50..100 - done.
✓ /media/sf_SCENARE/height/BRUM83_h.tif
300...30........60...401040.....70......5050....20...80.......606090.....30.....70..70....40...80...80..50.......90.90..60......70..100 - done.
✓ /media/sf_SCENARE/height/BRUM84_h.tif
0.80......10.90.....20...30...100 - done.
✓ /media/sf_SCENARE/height/BRUM92_h.tif
0..40.10....20...30...40...50.100 - done.
.✓ /media/sf_SCENARE/height/BRUM90_h.tif
.60...70.0...80....90...50..100 - done.
✓ /media/sf_SCENARE/height/BRUM95_h.tif
.0..10....60....10..20..70......30.2080......100 - done.
✓ /media/sf_SCENARE/height/BRUM93_h.tif
..0.90.30.40..........5010..40.....20.60....50..30.70........4060.....5080.....60..70...70....80..90....90......80.100 - done.
✓ /media/sf_SCENARE/height/BRUM94_h.tif
0.....9010......100 - done.
✓ /media/sf_SCENARE/height/HODO02_h.tif
20.0....30...10..40....20..50....30...60.100 - done.
✓ /media/sf_SCENARE/height/HODO00_h.tif
.040......50.70...

Now let's check the heights via minimum/maximum (ignoring nodata value -9999):

In [4]:
! for f in /media/sf_SCENARE/height/*_h.tif; do pkinfo -nodata -9999 -i "$f" -f -mm; done > /media/sf_SCENARE/height_mm.txt
! cat /media/sf_SCENARE/height_mm.txt

 --input /media/sf_SCENARE/height/BRUM50_h.tif -min -3.4e+38 -max 41.3123 
 --input /media/sf_SCENARE/height/BRUM51_h.tif -min -3.4e+38 -max 50.4174 
 --input /media/sf_SCENARE/height/BRUM60_h.tif -min -37.525 -max 53.8082 
 --input /media/sf_SCENARE/height/BRUM61_h.tif -min -3.4e+38 -max 71.9725 
 --input /media/sf_SCENARE/height/BRUM62_h.tif -min -3.4e+38 -max 94.8433 
 --input /media/sf_SCENARE/height/BRUM63_h.tif -min -3.4e+38 -max 47.3333 
 --input /media/sf_SCENARE/height/BRUM70_h.tif -min -20.6258 -max 45.9798 
 --input /media/sf_SCENARE/height/BRUM71_h.tif -min -85.5765 -max 60.7241 
 --input /media/sf_SCENARE/height/BRUM72_h.tif -min -43.8448 -max 60.3226 
 --input /media/sf_SCENARE/height/BRUM73_h.tif -min -3.4e+38 -max 87.2979 
 --input /media/sf_SCENARE/height/BRUM74_h.tif -min -3.4e+38 -max 71.3427 
 --input /media/sf_SCENARE/height/BRUM80_h.tif -min -78.1104 -max 73.9842 
 --input /media/sf_SCENARE/height/BRUM81_h.tif -min -203.103 -max 45.0692 
 --input /media/sf_SCENARE

**All the tiles now have minimum values far below zero!!!** What happened? Is our data really that bad?
Let’s open the most critical tiles and take a closer look.

In [69]:
%%bash
echo "5 smallest min"
awk '{print $4, $2}' /media/sf_SCENARE/height_mm.txt | sort -k1,1g | awk '!seen[$1]++' | head -5

echo
echo "5 largest max"
awk '{print $6, $2}' /media/sf_SCENARE/height_mm.txt | sort -k1,1gr | head -5

5 smallest min
-3.4e+38 /media/sf_SCENARE/height/BRUM50_h.tif
-556.334 /media/sf_SCENARE/height/UBRO78_h.tif
-489.454 /media/sf_SCENARE/height/KYJO38_h.tif
-439.923 /media/sf_SCENARE/height/UHRA97_h.tif
-373.532 /media/sf_SCENARE/height/UBRO92_h.tif

5 largest max
242.868 /media/sf_SCENARE/height/BRUM94_h.tif
222.421 /media/sf_SCENARE/height/KYJO39_h.tif
216.774 /media/sf_SCENARE/height/KYJO29_h.tif
204.582 /media/sf_SCENARE/height/VELV23_h.tif
141.166 /media/sf_SCENARE/height/UBRO98_h.tif


Let's classify our height values in the viewer:
red - large negative values, incl. -3.4e+38, 
yellow - lower errors (from -1 to 0), 
green - low objwcts (0  -1 m), 
blue - bushes (1 - 3 m), 
pink - trees and buildings (>3 m).
![UBRO50](pics/ubro50.png)
![BRUM94](pics/brum94.png)
![UBRO78](pics/ubro78.png)

As we can see, the value -3.4e+38 appears only along the border of Czechia. Our DSM and DEM were probably produced with different buffer widths from the Czech border, so the calculation returned the lowest possible value (-3.4e+38) in areas where either the DSM or DEM was missing.

Other large negative values (-556.334 … -1.000) mostly occur in shadowed areas at the forest edge or in forest gaps, probably due to stereoscopic processing errors or sensor issues. Values from -1 to 0 are located in low-vegetation or bare-ground areas and most likely reflect the vertical error of the input data, which is up to 30 cm in open land and up to 70 cm in forested areas. We can ignore these values by treating them as low vegetation (< 1 m).

We can also see that almost all “tree” and “bush” vegetation aligns very well with the orthophotos, so our data quality is actually quite good.

The height of the tallest tree in the Czech Republic is approximately 67 m (see: https://www.silvarium.cz/lesnictvi/nejvyssi-strom-v-ceske-republice-byl-po-deseti-letech-znovu-premeren) Therefore, heights above 67 m likely correspond to buildings or to errors/outliers, which are also outside the scope of our analysis.

First, let’s remove the -3.4e+38 values by setting them to NoData. Then we will reclassify our rasters into classes 0, 1, 2, and 3 as follows:
- Class 0 – large negative values (we keep them as 0, not NoData and not 1, so we can later fill them using neighbouring values);
- Class 1 – “other / low vegetation”, including values from -1 to 0 and values > 67 m;
- Class 2 – bush vegetation (1–3 m);
- Class 3 – trees (3–67 m).

In [5]:
%%bash

# make mask and apply nodata to -3.4e+38

IN_DIR="/media/sf_SCENARE/height"
OUT_DIR="/media/sf_SCENARE/MASK"

for f in "$IN_DIR"/*_h.tif; do
  [[ -f "$f" ]] || continue
  name="$(basename "$f" .tif)"
  pkgetmask \
    -i "$f" \
    -o "$OUT_DIR/${name}_mask.tif" \
    --min -600 \
    --data 1 --nodata 0 \
    -co COMPRESS=LZW -co TILED=YES
  echo "✓ $OUT_DIR/${name}_mask.tif"
done

0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM50_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM51_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM60_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM61_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM62_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM63_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM70_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM71_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/MASK/BRUM72_h_mask.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /

In [6]:
%%bash

IN_DIR="/media/sf_SCENARE/height"
MASK_DIR="/media/sf_SCENARE/MASK"
OUT_DIR="/media/sf_SCENARE/height_masked"
NODATA=-9999

mkdir -p "$OUT_DIR"

# Loop over all mask rasters *_mask.tif
for mask in "$MASK_DIR"/*_mask.tif; do
  [[ -f "$mask" ]] || continue   # skip if glob doesn't match anything
  
  name=$(basename "$mask" _mask.tif)

  in_tif="$IN_DIR/${name}.tif"
  out_tif="$OUT_DIR/${name}.tif"

  echo "Masking $name → $out_tif ..."

  # Apply mask: pixels with 0 in mask → nodata (-9999), 1 → keep
  pksetmask \
    -i "$in_tif" \
    -m "$mask" \
    --msknodata 0 \
    --nodata "$NODATA" \
    -o "$out_tif" \
    -co COMPRESS=LZW -co TILED=YES

  echo "✓ $out_tif"
done

echo "All masked rasters written to $OUT_DIR"

Masking BRUM60_h → /media/sf_SCENARE/height_masked/BRUM60_h.tif ...
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_masked/BRUM60_h.tif
Masking BRUM61_h → /media/sf_SCENARE/height_masked/BRUM61_h.tif ...
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_masked/BRUM61_h.tif
Masking BRUM62_h → /media/sf_SCENARE/height_masked/BRUM62_h.tif ...
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_masked/BRUM62_h.tif
Masking BRUM63_h → /media/sf_SCENARE/height_masked/BRUM63_h.tif ...
0...10...20...30...40...50...60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_masked/BRUM63_h.tif
All masked rasters written to /media/sf_SCENARE/height_masked


**pkreclass**, when using the histogram approach, **cannot work with float data types**. Since we don’t need exact height values for the tree/bush mask and are going to reclassify the height raster anyway, we can safely convert it to the Int16 data type.

**Caution!** Converting to Int16 will simply truncate the decimal part (cut everything after the decimal point); it is not the same as mathematical rounding. This means:

Values from -1.999999999 to -1.000000000 become -1  
Values from -0.999999999 to 0.999999999 become 0  
Values from 1.000000000 to 1.999999999 become 1, and so on  

For our purposes, this approach is acceptable.

So, let’s first convert the raster datatype to Int16 and then create a code.txt with the reclassification rules as follows:

< 1 → 0  
0 → 1  
1 → 2  
2 → 2  
3 … 67 → 3  
68 and higher → 1

In [7]:
%%bash
IN_DIR="/media/sf_SCENARE/height_masked"
OUT_DIR="/media/sf_SCENARE/height_masked_int16"

mkdir -p "$OUT_DIR"

for f in "$IN_DIR"/*.tif; do gdal_translate -ot Int16 -a_srs EPSG:5514 -a_nodata -9999 -stats -co COMPRESS=DEFLATE "$f" "$OUT_DIR/$(basename "$f")"; done
echo "All done"

Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 5000, 4000
0...10...20...30...40...50...60...70...80...90...100 - done.

In [10]:
! cat /media/sf_SCENARE/height_masked_int16/code.txt # to see the code for reclassifying

-9999	-9999
-556	0
-555	0
-554	0
-553	0
-552	0
-551	0
-550	0
-549	0
-548	0
-547	0
-546	0
-545	0
-544	0
-543	0
-542	0
-541	0
-540	0
-539	0
-538	0
-537	0
-536	0
-535	0
-534	0
-533	0
-532	0
-531	0
-530	0
-529	0
-528	0
-527	0
-526	0
-525	0
-524	0
-523	0
-522	0
-521	0
-520	0
-519	0
-518	0
-517	0
-516	0
-515	0
-514	0
-513	0
-512	0
-511	0
-510	0
-509	0
-508	0
-507	0
-506	0
-505	0
-504	0
-503	0
-502	0
-501	0
-500	0
-499	0
-498	0
-497	0
-496	0
-495	0
-494	0
-493	0
-492	0
-491	0
-490	0
-489	0
-488	0
-487	0
-486	0
-485	0
-484	0
-483	0
-482	0
-481	0
-480	0
-479	0
-478	0
-477	0
-476	0
-475	0
-474	0
-473	0
-472	0
-471	0
-470	0
-469	0
-468	0
-467	0
-466	0
-465	0
-464	0
-463	0
-462	0
-461	0
-460	0
-459	0
-458	0
-457	0
-456	0
-455	0
-454	0
-453	0
-452	0
-451	0
-450	0
-449	0
-448	0
-447	0
-446	0
-445	0
-444	0
-443	0
-442	0
-441	0
-440	0
-439	0
-438	0
-437	0
-436	0
-435	0
-434	0
-433	0
-432	0
-431	0
-430	0
-429	0
-428	0
-427	0
-426	0
-425	0
-424	0
-423	0
-422	0
-421	0
-420	0
-419	0
-418	0
-417	0
-416	0
-

In [11]:
%%bash

# perform reclass with xargs

IN_DIR="/media/sf_SCENARE/height_masked_int16"
OUT_DIR="/media/sf_SCENARE/height_reclass"
CODE="/media/sf_SCENARE/height_masked_int16/code.txt"

mkdir -p "$OUT_DIR"

# make variables visible in the xargs-spawned shells
export OUT_DIR CODE

find "$IN_DIR" -maxdepth 1 -type f -name '*.tif' -print0 | \
xargs -0 -P 4 -I{} bash -c '
  f="$1"
  out="$OUT_DIR/$(basename "$f")"

  pkreclass -co COMPRESS=DEFLATE -co ZLEVEL=9 \
    -code "$CODE" \
    -i "$f" \
    -o "$out"

  echo "✓ $out"
' _ {}

echo "All done"

0000..........10.10...10..10.20.......20....302020........40..30..30....50.......404060...30.......507050...........40806060........90......7070.100 - done.
✓ /media/sf_SCENARE/height_reclass/VELV93_h.tif
..50.....080..80......10.90.....60.20....100 - done.
✓ /media/sf_SCENARE/height_reclass/VELV92_h.tif
90...030......40......50100 - done.
.✓ /media/sf_SCENARE/height_reclass/VELV90_h.tif
70...0.6010......70......80.80.10....20..90........100 - done.
20✓ /media/sf_SCENARE/height_reclass/VELV94_h.tif
90..0..30......30...100 - done.
✓ /media/sf_SCENARE/height_reclass/VELV91_h.tif
.100...40.40......10.50.....20....20....50.30.60..30.....40......60..504070.........7060....50.....7080...80.....60..80..90.....90.90........100 - done.
✓ /media/sf_SCENARE/height_reclass/VKLO58_h.tif
100 - done.
✓ /media/sf_SCENARE/height_reclass/VKLO56_h.tif
.700.0100 - done.
.✓ /media/sf_SCENARE/height_reclass/VKLO55_h.tif
..10.0.....20...80..30.....40......10.105090..........6020.....100 - done.
✓ /media/sf_S

Let’s have a look at the histograms of several tiles to check how large the “0” class is. Some tiles contain class 0, while others do not.

In [13]:
! pkstat -hist -i /media/sf_SCENARE/height_reclass/BRUM91_h.tif | grep -v " 0" 

0 77134
1 10598508
2 613298
3 8711060


In [14]:
! pkstat -hist -i /media/sf_SCENARE/height_reclass/BRUM95_h.tif | grep -v " 0" 

-9999 19917364
1 586
2 263
3 81787


In [15]:
! pkstat -hist -i /media/sf_SCENARE/height_reclass/BRUM74_h.tif | grep -v " 0" 

-9999 14167988
0 44348
1 904430
2 272024
3 4611210


In [16]:
# let's check the data type.
! gdalinfo /media/sf_SCENARE/height_reclass/BRUM91_h.tif | grep "Type"

Band 1 Block=5000x1 Type=Int16, ColorInterp=Gray


Now we should temporarily set class 0 as NoData so that we can later fill these gaps using the **pkfillnodata**
tool.

In [17]:
%%bash
mkdir -p /media/sf_SCENARE/height_reclass_0
mkdir -p /media/sf_SCENARE/height_reclass_filled

OUT_DIR="/media/sf_SCENARE/height_reclass_0"
export OUT_DIR

find /media/sf_SCENARE/height_reclass -maxdepth 1 -type f -name '*.tif' -print0 | \
xargs -0 -P 4 -I{} bash -c '
  f="$1"
  gdal_translate -a_nodata 0 "$f" "$OUT_DIR/$(basename "$f")"
  echo "✓ $OUT_DIR/$(basename "$f")"
' _ {}

echo "All done"

Input file size is 5000, 4000
0Input file size is 5000, 4000
0Input file size is 5000, 4000
0Input file size is 5000, 4000
0...10...20.....10...20.....10...20.....10...20...30...40...50.30...40...50.30...40...50.30...40...50...60...70.....60...70.....60...70.....60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VELV93_h.tif
.80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VELV94_h.tif
Input file size is 5000, 4000
0.80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VELV91_h.tif
.80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VELV92_h.tif
Input file size is 5000, 4000
0Input file size is 5000, 4000
0Input file size is 5000, 4000
0...10...20.....10...20.....10...20.....10...20...30...40...50.30...40...50.30...40...50.30...40...50...60...70.....60...70.....60...70.....60...70...80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VKLO55_h.tif
.80...90...100 - done.
✓ /media/sf_SCENARE/height_reclass_0/VKLO58_h.tif
Input file size is

In [3]:
%%bash
IN_DIR="/media/sf_SCENARE/height_reclass_0"
OUT_DIR="/media/sf_SCENARE/height_reclass_filled"

export IN_DIR OUT_DIR

find "$IN_DIR" -maxdepth 1 -type f -name '*.tif' -print0 | \
xargs -0 -P 4 -I{} bash -c '
  f="$1"
  pkfillnodata -i "$f" -m "$f" -o "$OUT_DIR/$(basename "$f")" -d 250  
# 250 is a maximum number of pixels to search in all directions to find values to interpolate from. 
# Some holes were quite large!
  echo "✓ $OUT_DIR/$(basename "$f")"
' _ {}

echo "All done"

0000............10101010............20202020............30303030............40404040............505050.....6050..60.....7070......8080........9090.........60....6070......8070.....100 - done.
✓ /media/sf_SCENARE/height_reclass_filled/VELV92_h.tif
90100 - done.
✓ /media/sf_SCENARE/height_reclass_filled/VELV91_h.tif
..80.....90...0.0....10..10....20..20....30..30....40..40....50100 - done.
.✓ /media/sf_SCENARE/height_reclass_filled/VELV93_h.tif
..60..100 - done.
.✓ /media/sf_SCENARE/height_reclass_filled/VELV94_h.tif
.70...8050...90......60...70...80...90...00.....10..10....20..20....30..30..100 - done.
✓ /media/sf_SCENARE/height_reclass_filled/VKLO55_h.tif
..40..40....50...50.100 - done.
✓ /media/sf_SCENARE/height_reclass_filled/VKLO56_h.tif
60....70...80....90..0....60..10....70.0.20.......80.10.30........902040.........3050.....40..60....50...60....70...80...9070...100 - done.
✓ /media/sf_SCENARE/height_reclass_filled/VKLO57_h.tif
...80...90...0.100 - done.
✓ /media/sf_SCENARE/height_

In [21]:
! gdalinfo /media/sf_SCENARE/height_reclass_filled/BRUM91_h.tif

Driver: GTiff/GeoTIFF
Files: /media/sf_SCENARE/height_reclass_filled/BRUM91_h.tif
       /media/sf_SCENARE/height_reclass_filled/BRUM91_h.tif.aux.xml
Size is 5000, 4000
Coordinate System is:
PROJCRS["S-JTSK / Krovak East North",
    BASEGEOGCRS["S-JTSK",
        DATUM["System of the Unified Trigonometrical Cadastral Network",
            ELLIPSOID["Bessel 1841",6377397.155,299.1528128,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4156]],
    CONVERSION["Krovak East North (Greenwich)",
        METHOD["Krovak (North Orientated)",
            ID["EPSG",1041]],
        PARAMETER["Latitude of projection centre",49.5,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8811]],
        PARAMETER["Longitude of origin",24.8333333333333,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8833]],
        PARAMETER["Co-latitude of cone axis",30.2881397527778

Holes (red) have dissapeared.
![holes](pics/hole1.png)
![no_holes](pics/hole2.png)

Let’s create a combined histogram for all tiles to check whether the 0 class is still present after refilling.

In [4]:
%%bash
ls /media/sf_SCENARE/height_reclass_filled/*.tif | xargs -n 1 -P 4 bash -c 'pkstat --hist -i "$0" > "$0.txt"'
awk '{count[$1]+=$2} END {for (i in count) print i, count[i]}' /media/sf_SCENARE/height_reclass_filled/*.txt | sort -n > /media/sf_SCENARE/height_reclass_filled/summed_hist.txt
echo "Done"

Done


In [5]:
! cat /media/sf_SCENARE/height_reclass_filled/summed_hist.txt | grep -v " 0"

-9999 539116034
1 2763694776
2 585817499
3 2391371691


We don't have "0" class anymore - all gaps were filled.

## Removing buildings from the mask
To remove buildings (objects 3–67 m in height) from the tree layer, we use the building footprint shapefile. Unfortunately, this dataset is not ideal: some buildings are missing, and the building shapes do not always align perfectly with the orthophoto, they are often narrower than the actual buildings visible in the imagery.

![buildings1](pics/build1.png)
![buildings2](pics/build2.png)

Therefore, we apply a 1 m buffer around the building polygons to remove, as many as possible, any remaining building fragments from the final mask.

In [48]:
%%bash
# rewrite into gpkg (because of huge size) and create buffers

ogr2ogr -f GPKG /media/sf_SCENARE/buildings_buf_temp.gpkg \
  /media/sf_SCENARE/buildings.shp \
  -dialect SQLite \
  -nln buildings_buf -overwrite -nlt MULTIPOLYGON \
  -sql "SELECT
           ST_SimplifyPreserveTopology(ST_Buffer(geometry, 1.0), 0.05) AS geometry,
           *
        FROM buildings"
echo "Done."

Done.


Now let's create the raster mask of buildings, burning "1" where buildings exist.

In [11]:
%%bash
# burn 1 where polygons exist
VEC="/media/sf_SCENARE/buildings_buf_temp.gpkg"             
OUT="/media/sf_SCENARE/buildings_mask.tif"

XMIN=-560000.000
YMIN=-1210000.000
XMAX=-487500.000
YMAX=-1166000.000

# assign nodata as 255 to preserve Byte data type to make the output smaller in size
gdal_rasterize -burn 1 -ot Byte -at -a_nodata 255 -l buildings_buf \
  -tr 0.5 0.5 -te $XMIN $YMIN $XMAX $YMAX \
  "$VEC" "$OUT"

echo "Mask is done"

0...10...20...30...40...50...60...70...80...90...100 - done.
Mask is done


Unfortunately, we cannot apply a single buildings mask to each tile of our classified rasters, because the pktools commands require all input data to have the same extent. Therefore, we also need to tile our buildings mask. For this, we use footprints.gpkg (the tile outlines) that we created earlier.

In [12]:
%%bash
VEC="/media/sf_SCENARE/footprints.gpkg"
OUTDIR="/media/sf_SCENARE/buildmask_tiles"

mkdir -p "$OUTDIR"

# 2) Collect all unique 'location' values from the shapefile (case-sensitive field name!)
mapfile -t LOCS < <(
  ogrinfo -ro -al -geom=no -q "$VEC" \
  | awk -F'= ' '/^[[:space:]]*location[[:space:]]*\(String\)[[:space:]]*=/{print $2}' \
  | sed 's/\r$//' | sort -u
)

# 3) Crop per polygon to tiles, naming outputs from the 'location' field
for loc in "${LOCS[@]}"; do
  name="$(basename "$loc" .tif)"
  out="$OUTDIR/${name}.tif"
  sql_val=${loc//\'/\'\'}  # escape quote if any

  gdalwarp -overwrite -of GTiff \
    -cutline "$VEC" -cwhere "location = '$sql_val'" \
    -crop_to_cutline \
    -r bilinear \
    -dstnodata 255 -s_srs EPSG:5514 -t_srs EPSG:5514 \
    -co COMPRESS=LZW -co TILED=YES -co BIGTIFF=YES \
    /media/sf_SCENARE/buildings_mask.tif "$out"
  echo "$out is done"
  done
echo "All done"

Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/buildings_mask.tif [1/1] : 0Using internal nodata values (e.g. 255) for image /media/sf_SCENARE/buildings_mask.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
/media/sf_SCENARE/buildmask_tiles/BRUM50.tif is done
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/buildings_mask.tif [1/1] : 0Using internal nodata values (e.g. 255) for image /media/sf_SCENARE/buildings_mask.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
/media/sf_SCENARE/buildmask_tiles/BRUM51.tif is done
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/buildings_mask.tif [1/1] : 0Using internal nodata values (e.g. 255) for image /media/sf_SCENARE/buildings_mask.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
/media/sf_SCENARE/buildmask_tiles/BRUM60.tif is done
Creating output file that is 5000P x 4000L.
Processing /media/sf_SCENARE/buildings_mask.tif [1

In [20]:
! gdalinfo /media/sf_SCENARE/buildmask_tiles/BRUM91.tif

Driver: GTiff/GeoTIFF
Files: /media/sf_SCENARE/buildmask_tiles/BRUM91.tif
       /media/sf_SCENARE/buildmask_tiles/BRUM91.tif.ovr
       /media/sf_SCENARE/buildmask_tiles/BRUM91.tif.aux.xml
Size is 5000, 4000
Coordinate System is:
PROJCRS["S-JTSK / Krovak East North",
    BASEGEOGCRS["S-JTSK",
        DATUM["System of the Unified Trigonometrical Cadastral Network",
            ELLIPSOID["Bessel 1841",6377397.155,299.1528128,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4156]],
    CONVERSION["Krovak East North (Greenwich)",
        METHOD["Krovak (North Orientated)",
            ID["EPSG",1041]],
        PARAMETER["Latitude of projection centre",49.5,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8811]],
        PARAMETER["Longitude of origin",24.8333333333333,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8833]],
        PARAMETER["Co

##### Final step
Now we are going to burn the value 1 into the reclassified rasters wherever buildings are present, according to the building mask. Also, we can keep output as Byte data type, because we already have 0 for nodata. Output should be of Byte data type with "2" for bushes, "3" for trees, "1" for other, "0" for nodata.

In [23]:
%%bash
export OUT_DIR="/media/sf_SCENARE/height_reclass_fin"
export IN_DIR="/media/sf_SCENARE/height_reclass_filled"
export MASK_DIR="/media/sf_SCENARE/buildmask_tiles"

mkdir -p "$OUT_DIR"

# Run pkcomposite in parallel (4 processes)
ls "${IN_DIR}"/*_h.tif \
  | xargs -P 4 -I{} bash -c '
      full="$1"
      name="$(basename "$full" _h.tif)"

      echo "Processing: $name"

      pkcomposite \
        -i "${IN_DIR}/${name}_h.tif" \
        -i "${MASK_DIR}/${name}.tif" \
        -o "${OUT_DIR}/${name}_mask.tif" \
        -ot Byte \
        -cr overwrite \
        -srcnodata 255 \
    ' _ {}

echo "All done"

Processing: BRUM50
Processing: BRUM61
Processing: BRUM51
Processing: BRUM60
0000............10101010............202020.20...........30303030...........40.404040...........50.5050.50.........60.60...6060........70.70...70..70..80.......90.80.....100 - done.
80Processing: BRUM62
080......90....90....1090....100 - done.
.Processing: BRUM63
..0100 - done.
Processing: BRUM70
.0..20.100 - done.
.Processing: BRUM71
.0...10.....3010......10....4020.20......20.....303050.........30...404060.........40.50..5070.........50..6060...80....70.......80.6070...90..90......100 - done.
Processing: BRUM72
..0..7080..100 - done.
Processing: BRUM73
..0.......108090.....10.......20100 - done.
90Processing: BRUM74
0....20.......30.100 - done.
Processing: BRUM80
10.0...30.....40....2010....40....50....2030...50.....60.....40.3060.....70.......504070....80........608050....90........709060......100 - done.
Processing: BRUM81
.0....70100 - done.
Processing: BRUM82
80..0.....10....9080......100 - done.
Processin

Let's check the metadata of any tile to be sure everything is correct:

In [26]:
! pkinfo -i /media/sf_SCENARE/height_reclass_fin/BRUM50_mask.tif -dx -dy -ot -mm -nodata 0
#! gdalinfo /media/sf_SCENARE/height_reclass_fin/BRUM50_mask.tif

--dx 0.5 --dy 0.5 -min 1 -max 3 --otype Byte 


As we can see from the final output (second image), **most** buildings have disappeared from the final tree/bush mask. However, some buildings remain (those that were not included in the building footprint vector file), as well as some low buildings, cars, fences, overpasses, tall crops (corn) and herbs etc. Since it is impossible to remove all these types of objects, we will instead try to mark them and exclude them from the training set for the deep learning model.
![with_buildings](pics/fin1.png)
![no_buildings](pics/fin2.png)

The final result for the full extent can be viewed here:
https://vukoz.maps.arcgis.com/apps/mapviewer/index.html?webmap=3f99560600f74056a9c43ec752a78ceb

![dont switch to Python](pics/giuseppe.png)